## GNN<span style="font-size: 1.3em; font-family: Comic Sans MS, Aptos (Body);">4</span>ID

This notebook demonstrates a comprehensive pipeline to convert raw PCAP files into graph data objects suitable for Graph Neural Network (GNN) models. The pipeline extracts flow-level information alongside packet-level details, ultimately producing two primary outputs:

1. <u>Extracted Flow-based Features with Packet-Level Information</u>: Detailed flow-based features are extracted, including comprehensive packet-level information.
2. <u>Graph Data Objects for GNN Models</u>: Flow-based features and packet-level details are transformed into individual graph data objects, suitable for graph-level predictions using GNN models.

This transformation enables the application of GNNs for advanced network traffic analysis and intrusion detection, leveraging the rich information from both flow and packet levels.

In [ ]:
# !pip install -r requirement.txt

In [ ]:
from Utility.Functions import *
from Utility.Additional_Features import *
import tarfile
import glob
import shutil
import subprocess
from tqdm import tqdm

### Extraction of Compressed PCAP Files from the CIC-IoT2023 Dataset

For demonstration purposes, we will utilize the CIC-IoT2023 dataset, one of the latest and most comprehensive datasets available for IoT network traffic analysis. You can access and download the dataset from the following link: [CIC-IoT2023 Dataset](https://www.unb.ca/cic/datasets/iotdataset-2023.html).

While this example uses the CIC-IoT2023 dataset, any dataset with labeled raw packets can be used. If your PCAP files are not labeled, you can use the tool available at [Payload Byte](https://github.com/Yasir-ali-farrukh/Payload-Byte) for labeling the PCAP files.


The Structure for the folder directory is as follows:

        /CIC IoT Dataset 2023
                └── All downloaded CIC-IOT2023 .gz files
    
        
        /CIC_IOT
        ├── Packet_Level_Data
        │          └── Extracted Pcap Files
        |
        └── Extracted_Flow_Features
            ├         └──Extracted Features from Pcap files
            │
            │── train
            │     └── Single File for Train_Dataset
            │── test
            │     └── Single File for Test_Dataset
            │
            └── processed
                    └── Generated Graph Objects

In [ ]:
# Provide path to the directory where Raw Pcap Files are downloaded
# The CIC-IoT2023 Dataset is availble in compressed .tar format
Directory = "F:\\CIC IoT Dataset 2023\\*.tar.gz"
# Path where you want the extracted PCAP files to be
Out_Directory = 'F:\\CIC_IOT\\Packet_Level_Data'

Compressed_files = glob.glob(Directory)

In [ ]:
for files in Compressed_files:
    file = tarfile.open(files) 
    file.extractall(Out_Directory)

### Renaming the PCAP Files

To facilitate easier differentiation between attack classes during the transformation into graph data objects, it is essential to rename the PCAP files appropriately.

**Alternatively:** You can generate a single file containing all data instances along with a "Label" column to categorize the instances, which has been done in the preprocessing of CIC-IOT2023 Dataset `Data_preprocessing_CIC-IoT2023.ipynb`

In [ ]:
# Tên file quyết định nhãn về sau. Bộ ánh xạ 34 tên thư mục/file CIC-IoT2023 -> '<Class>-<Sub>'
# nằm ở Utility/Schema.py::CIC_SUBTYPES và rename_files() dùng resolver KHÔNG phân biệt hoa/thường
# (resolve_class_subtype). Dict cũ có 3 lỗi trên Linux: key 'DDos-SlowLoris' (thư mục thật là
# 'DDoS-SlowLoris'), value 'Webbased-BrwserHijack' (b thường -> glob 'WebBased*' không thấy), và
# key 'Benign' không khớp file thật 'BenignTraffic*.pcap' (=> benign bị coi là attack).
from Utility.Schema import CIC_SUBTYPES
name_mapping = {k: f"{cls}-{short}" for k, (cls, short) in CIC_SUBTYPES.items()}
name_mapping


In [ ]:
## Function that renames the files to '<Class>-<Sub>_<n>.pcap' (resolver-based, case-insensitive)
rename_files(Out_Directory)


### Extracting Features from PCAP Files

Extraction of flow-level features along with their respective packet-level features from PCAP files.

The features are extracted using the `Feature_extractor_flow_packet_combined.py` script. These features can be utilized for various purposes beyond creating graph objects, as they offer complete information about each flow along with its associated packet details.


In [ ]:
import os, sys
directory = os.path.join(Out_Directory, "**", "*.pcap*")
List_of_PCAP = [f for f in glob.glob(directory, recursive=True) if os.path.isfile(f)]   # files only (a folder named "pcap" would match "*pcap")
print(List_of_PCAP)
assert_all_resolvable(List_of_PCAP)   # every pcap must map to a CIC-IoT2023 class (else: fix the name)
Out_path = 'F:/CIC_IOT/Extracted_Flow_Features/'  # Directory path where the extracted flow+packet csv will be stored
feature_Extractor = 'Utility/Feature_extractor_flow_packet_combined.py'
DELETE_PCAP = False   # the original notebook deleted every pcap after extraction; keep them by default
for single_pcap_file in tqdm(List_of_PCAP):
    print("Reading File: ", os.path.basename(single_pcap_file))
    # Run the extractor on the command line: NFStream's multiprocessing misbehaves inside a notebook.
    completed_process = subprocess.run([sys.executable, feature_Extractor, single_pcap_file, Out_path,
                                        '--out-name', canonical_stem(single_pcap_file)],
                                       capture_output=True)
    if completed_process.returncode != 0:
        raise RuntimeError(completed_process.stderr.decode(errors="ignore")[-2000:])
    if DELETE_PCAP:
        os.remove(single_pcap_file)


### Additional Features Based on Temporal Information (Explainable Features)

`additional_features()` adds the 28 rolling-window columns of the authors (per destination IP and per
src-dst pair, window = 350 previous flows, `Utility/Additional_Features.py`), `packet_size_variation`
and the 7 one-hot columns -> 82 flow features, identical to the published Google-Drive CSVs.

It runs on the **whole, time-ordered raw file, before the train/test split**: the features are
label-free and only look backwards, so a later temporal split cannot leak, and train and test keep
the same feature semantics. The raw csv is not overwritten (output goes to `features/`).
Options: `window_unit='time'` (e.g. `window='60s'`, paper Algorithm 1 wording), `'packets'`,
`count_mode='packets'` (count packets instead of flows for UDP/TCP/ICMP/HTTP/DNS), `schema='table1'`.


In [ ]:
Extracted_Features_Files = [f for f in glob.glob(os.path.join(Out_path, "*.csv"))
                            if not os.path.basename(f).startswith("df_class_8")]   # per-pcap raw csv only
Features_path = os.path.join(Out_path, "features")     # raw csv stays untouched
for file in tqdm(Extracted_Features_Files):
    additional_features(file, out_file=os.path.join(Features_path, os.path.basename(file)))   # window=350 flows, author schema (82)
import pandas as pd
pd.read_csv(os.path.join(Features_path, os.path.basename(Extracted_Features_Files[0])), nrows=2).shape


### Transformation into Graph Data Objects
Utilizing the extracted flow-level features along with their respective packets, the data object created is a heterogeneous graph consisting of two different types of nodes and two different types of edges. The nodes are:

1. Flow Node: Contains all flow-level statistical features.
2. Packet Node: Contains payload information transformed into byte-wise values.

The two different edges are:

1. Contain Edge: Links Flow Nodes and Packet Nodes along with some features.
2. Link Edge: Links Packet Nodes together with t-delta as its attribute.

In [ ]:
## Dictionary for classifying Classes and Assigning them Class number for reference
Dict_x = {'Benign': 0 , 
          'WebBased': 1, 
          'Spoofing': 2,
          'Recon' : 3,
          'Mirai' : 4,
          'Dos' : 5,
          'DDos' : 6,
          'BruteForce': 7
         }

## Directory where graph data will be stored
dir = "F:/CIC_IOT/Extracted_Flow_Features/"
## Directory where CSV files(Extracted Flow-level and packet-level inforamtion) is stored
Files = [os.path.join(Out_path, "df_class_8_train.csv")]   # produced by Data_preprocessing_CIC-IoT2023.ipynb (build_class8_csvs) ## This will list all the files from which graph data objects will be created.
## Uncomment for the generation of test graph objects.
# Files = glob.glob(Out_path+"/test/*.csv")

Since the CIC-IoT2023 dataset is large and has imbalanced instances of classes, we have performed data processing (over/under sampling) to achieve a balanced dataset for ease of training and to address the imbalance problem. To follow the pre-processing steps, please refer to the notebook: `Data_preprocessing_CIC-IoT2023.ipynb`. Also in the provided preprocessing notebook, we have compiled all classes data into one single file for ease of use and reproducibility. The processed file can be downloaded through the following link: [Processed_CIC-IoT2023](https://drive.google.com/drive/folders/1FiZh87vvCZF3gX1Fnj9iTB4j74u-nuR6?usp=sharing) 




In [ ]:
## Graph objects are numbered from 0: remove .pt files of a previous run first, otherwise
## NIDSDataset counts stale files (built from other data / another schema) as part of the dataset.
for _old in glob.glob(os.path.join(dir, "processed", "*.pt")):
    os.remove(_old)
## Generation of graph data obejcts.
data_Hetero = NIDSDataset(root=dir, label_dict=Dict_x, filename=Files, skip_processing=False, test=False, single_file=True) 
# Here we have utilized the sinlge file created through the `Data_preprocessing_CIC-IoT2023.ipynb`, however we can also utilize indiviudal files without any preprocessing directly.

PARAMETERS 

- **root** (`str`): Root directory where the graph objects should be saved.

- **label_dict** (`Dict`): Dictionary for assigning labels to each attack class.

- **filename** (`List[str]`): List of CSV file paths to be used for the development of graph objects.

- **skip_processing** (`bool`): If set to `True`, skips the generation of graph objects and utilizes the ones present in the root directory. (default: `False`)

- **test** (`bool`): If set to `True`, generates data objects for testing by creating data objects with a test suffix. (default: `False`)

- **single_file** (`bool`): If set to True, the provided CSV files is a single file with Label column within CSV.  (default: `False`)   
